In [1]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.messages import HumanMessage,AIMessage,SystemMessage

load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [2]:
llm=ChatGroq(model='openai/gpt-oss-120b')

In [3]:

DB_NAME=r"E:\chatbot\restaurant-booking-systems\restaurant_bookings.db"

In [4]:
import sqlite3

In [5]:
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
drop table restaurant_bookings
""")

available_tables = cursor.fetchall()

conn.close()

In [6]:
def create_table():

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS restaurant_bookings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            guests INTEGER NOT NULL,
            date TEXT NOT NULL,
            time TEXT NOT NULL,
            table_id TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()

    print("Table created successfully!")

create_table()

Table created successfully!


In [7]:
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
INSERT INTO restaurant_bookings
        (name, guests, date, time, table_id)
        VALUES ('azra', 2, '10-12-2026', '7:00 pm', 'table-1')
        """)

conn.commit()
available_tables = cursor.fetchall()

conn.close()

In [8]:
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
    SELECT table_id
    FROM restaurant_tables
    WHERE table_id NOT IN (
        SELECT table_id
        FROM restaurant_bookings
        WHERE date = ?
        AND time = ?
    )
""", ("10-12-2026", "7:00 pm"))

available_tables = cursor.fetchall()

conn.close()

print(available_tables)

[('table-2',), ('table-3',), ('table-4',), ('table-5',)]


In [9]:
def create_tables():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS restaurant_tables (
            table_id TEXT PRIMARY KEY,
            is_booked BOOLEAN DEFAULT 0
        )
    """)

    conn.commit()
    conn.close()

create_tables()

In [10]:
def insert_tables():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    tables = [
        ('table-1', False),
        ('table-2', False),
        ('table-3', False),
        ('table-4', False),
        ('table-5', False)
    ]

    cursor.executemany(
        """INSERT OR IGNORE INTO restaurant_tables
        (table_id, is_booked)
        VALUES (?, ?)""",
        tables
    )

    conn.commit()
    conn.close()

insert_tables()

In [11]:
def create_table():

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS restaurant_bookings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            guests INTEGER NOT NULL,
            date TEXT NOT NULL,
            time TEXT NOT NULL,
            table_id TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()

    print("Table created successfully!")

create_table()

Table created successfully!


In [12]:
def get_available_tables(booking_date, booking_time):

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT table_id
        FROM restaurant_tables
        WHERE table_id NOT IN (
            SELECT table_id
            FROM restaurant_bookings
            WHERE date = ?
            AND time = ?
        )
    """, (booking_date, booking_time))

    available_tables = cursor.fetchall()

    conn.close()

    return available_tables

In [13]:



def save_booking(name: str, guests: int, date: str, time: str, table_id: str):

    conn = sqlite3.connect(DB_NAME)

    try:
        cursor = conn.cursor()

        cursor.execute("""
            INSERT INTO restaurant_bookings
            (name, guests, date, time, table_id)
            VALUES (?, ?, ?, ?, ?)
        """, (name, guests, date, time, table_id))

        booking_id = cursor.lastrowid
        conn.commit()

        return booking_id

    finally:
        conn.close()


In [14]:

def get_bookings():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor = conn.execute("""
        SELECT *
        FROM bookings
        ORDER BY id DESC
    """)

    bookings = cursor.fetchall()
    conn.close()
    return bookings



In [15]:


@tool
def book_table(
    name: str,
    guests: int,
    date: str,
    time: str,
    table_id: str
):
    """Book a restaurant table if it is available for the requested date and time."""

    conn = sqlite3.connect(DB_NAME)

    try:
        cursor = conn.cursor()

        cursor.execute("BEGIN IMMEDIATE")

        cursor.execute("""
            SELECT table_id
            FROM restaurant_tables
            WHERE table_id = ?
        """, (table_id,))

        if not cursor.fetchone():
            return f"{table_id} does not exist."

        cursor.execute("""
            SELECT id
            FROM restaurant_bookings
            WHERE date = ? AND time = ? AND table_id = ?
        """, (date, time, table_id))

        if cursor.fetchone():
            return f"Sorry, {table_id} is already booked on {date} at {time}."

        cursor.execute("""
            INSERT INTO restaurant_bookings
            (name, guests, date, time, table_id)
            VALUES (?, ?, ?, ?, ?)
        """, (name, guests, date, time, table_id))

        booking_id = cursor.lastrowid
        conn.commit()

        return f"Booking confirmed! Reservation ID: {booking_id}"

    except Exception:
        conn.rollback()
        raise

    finally:
        conn.close()

@tool
def check_table_availability(date: str, time: str, table_id: str) -> str:
    """Check whether a restaurant table is available on a specific date and time."""

    conn = sqlite3.connect(DB_NAME)

    try:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT table_id
            FROM restaurant_tables
            WHERE table_id = ?
        """, (table_id,))

        if not cursor.fetchone():
            return f"{table_id} does not exist."

        cursor.execute("""
            SELECT id
            FROM restaurant_bookings
            WHERE date = ? AND time = ? AND table_id = ?
        """, (date, time, table_id))

        if cursor.fetchone():
            return f"Sorry, {table_id} is already booked on {date} at {time}."

        return f"{table_id} is available on {date} at {time}."

    finally:
        conn.close()


@tool
def view_bookings():
    """Retrieve existing restaurant reservations."""

    bookings=get_bookings()

    if not bookings:
        return "No bookings found."

    return str(bookings)




In [16]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware


In [17]:

agent = create_agent(
    model=llm,

    tools=[
        book_table,
        view_bookings,
        check_table_availability
    ],

    checkpointer=InMemorySaver(),

    system_prompt = """
You are a restaurant reservation assistant.

IMPORTANT RULES:

1. Never invent table IDs.
2. Always retrieve valid table IDs from the database.
3. Valid table IDs are retrieved using get_valid_table_ids().
4. The restaurant currently has only these tables:
   table-1, table-2, table-3, table-4, table-5.
5. Never suggest T1, T2, T3, T4, T5 or any other ID
   unless that exact ID exists in the database.
6. Before confirming a reservation, check whether the
   selected table is available for the requested date and time.
7. If the user does not have a table preference, select an
   available table from the database.
8. Ask for final approval before saving the reservation.
""",

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_table": {
                    "allowed_decisions": ["approve", "reject"]
                },

                "view_bookings": False,

                "check_table_availability": False
            }
        )
    ]
)


In [18]:

from langgraph.types import Command

config = {"configurable": {"thread_id": "chat4"}}

while True:
    user_input = input("YOU: ").strip()

    if user_input.lower() == "exit":
        print("BOT: Goodbye!")
        break

    # First message or continue the existing conversation
    if "__interrupt__" not in result if "result" in globals() else True:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]},
            config=config
        )

    # Handle booking approval
    while "__interrupt__" in result:
        while True:
            user_input = input("YOU (yes/no): ").lower().strip()

            if user_input in ["yes", "y", "approve"]:
                decision = {"type": "approve"}
                break

            elif user_input in ["no", "n", "reject"]:
                decision = {
                    "type": "reject",
                    "message": "User rejected the booking"
                }
                break

            else:
                print("Please enter yes or no.")

        result = agent.invoke(
            Command(resume={"decisions": [decision]}),
            config=config
        )

    print("\nBOT:", result["messages"][-1].content)



BOT: Hello! Welcome to **The Gourmet Spot**. I'd be happy to help you reserve a table.

Could you please let me know:

- The date you’d like to dine (e.g., 2026‑11‑05)  
- The time you’d like to be seated (e.g., 7:30 PM)  
- How many guests will be joining you  
- The name the reservation should be under  
- If you have a preference for a specific table (e.g., table‑2)  

Once I have those details, I’ll check availability and confirm the reservation for you.

BOT: Great news, **Azra**! Table‑1 is available on **December 10, 2026 at 7:00 PM** for your party of 2.

Would you like me to go ahead and confirm this reservation?

BOT: Got it! You’d like to reserve **Table‑1** for **2 guests** on **October 12, 2026 at 7:00 PM** under the name **Azra**.

Would you like me to go ahead and confirm this reservation?

BOT: Your reservation is confirmed! 🎉  

**Reservation ID:** 2  
**Name:** Azra  
**Date:** October 12, 2026  
**Time:** 7:00 PM  
**Table:** table-1  
**Guests:** 2  

We look forwa